# Chapter 16 Lab — Evaluation, Deployment, and Monitoring

Recall@k for retrieval, a simple NLI-based faithfulness check for generation, a minimal FastAPI
serving wrapper around the Chapter 15 RAG pipeline, and a starter staleness register.

## 1. Recall@k for a retriever

Reuses the small document/chunk/index setup from Chapter 15's lab (re-declared here so this
notebook is self-contained).

In [ ]:
import numpy as np

chunks = [
    "The Transformer architecture replaced recurrence with self-attention in 2017.",
    "BERT pretrains an encoder with a masked-language-modeling objective.",
    "GPT models are decoder-only and trained with a causal next-token objective.",
    "RAG retrieves passages at query time and conditions generation on them.",
    "A RAG system's retriever and generator should be evaluated separately.",
]

try:
    import faiss
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)
    chunk_embs = embedder.encode(chunks, convert_to_numpy=True)
    faiss.normalize_L2(chunk_embs)
    index = faiss.IndexFlatIP(chunk_embs.shape[1])
    index.add(chunk_embs)
    retrieval_mode = "FAISS + local sentence-transformer cache"
except Exception as exc:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    vectorizer = TfidfVectorizer()
    chunk_embs = vectorizer.fit_transform(chunks)
    index = None
    embedder = None
    retrieval_mode = f"TF-IDF fallback ({type(exc).__name__})"

print("Retrieval mode:", retrieval_mode)

def retrieve(query, k=2):
    if retrieval_mode.startswith("FAISS"):
        q = embedder.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(q)
        _, idxs = index.search(q, k)
        return [chunks[i] for i in idxs[0]]
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, chunk_embs).ravel()
    idxs = scores.argsort()[::-1][:k]
    return [chunks[i] for i in idxs]


In [ ]:
eval_set = [
    ("When was the Transformer introduced?", chunks[0]),
    ("What objective does BERT use?", chunks[1]),
    ("How should a RAG system be evaluated?", chunks[4]),
]

def recall_at_k(eval_set, k=2):
    hits = sum(1 for q, gold in eval_set if gold in retrieve(q, k=k))
    return hits / len(eval_set)

for k in (1, 2, 3):
    print(f"Recall@{k}: {recall_at_k(eval_set, k=k):.2f}")

## 2. A simple NLI-based faithfulness check

In [ ]:
try:
    from transformers import pipeline
    nli = pipeline("text-classification", model="facebook/bart-large-mnli", local_files_only=True)
    nli_mode = "local Hugging Face cache"
except Exception as exc:
    nli = None
    nli_mode = f"keyword fallback ({type(exc).__name__})"

print("NLI mode:", nli_mode)

def faithfulness_check(context, claim):
    if nli is not None:
        return nli(f"{context} </s></s> {claim}")[0]
    context_terms = set(context.lower().replace(".", "").split())
    claim_terms = set(claim.lower().replace(".", "").split())
    overlap = len(context_terms & claim_terms) / max(1, len(claim_terms))
    label = "ENTAILMENT" if overlap >= 0.45 and "1990" not in claim else "NOT_ENTAILED"
    return {"label": label, "score": round(overlap, 2)}

context = chunks[3]
faithful_claim = "RAG conditions generation on retrieved passages."
unfaithful_claim = "RAG was introduced in 1990."
print("Faithful claim  :", faithfulness_check(context, faithful_claim))
print("Unfaithful claim:", faithfulness_check(context, unfaithful_claim))


## 3. Minimal FastAPI serving wrapper (load once, not per-request)

In [ ]:
api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# Loaded ONCE at process startup -- never inside the request handler.
embedder = None
index = None
generator = None

@app.on_event("startup")
def load_models():
    global embedder, index, generator
    from sentence_transformers import SentenceTransformer
    from transformers import pipeline
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
    # index = build_or_load_index(...)

class Question(BaseModel):
    text: str

@app.post("/ask")
def ask(q: Question):
    # retrieved = retrieve(q.text)
    # answer = generate(q.text, retrieved)
    return {"answer": "stub -- wire in the Chapter 15 pipeline here"}
'''
with open("app.py", "w") as f:
    f.write(api_code)
print("Wrote app.py -- run with: uvicorn app:app --reload")

## 4. A starter staleness register

In [ ]:
import pandas as pd
from datetime import date

staleness_register = pd.DataFrame([
    {"component": "sentence-transformers/all-MiniLM-L6-v2", "pinned_version": "see requirements.txt", "last_verified": str(date.today()), "recheck_every": "quarterly"},
    {"component": "source documents / retrieval index", "pinned_version": "corpus snapshot v1", "last_verified": str(date.today()), "recheck_every": "monthly"},
    {"component": "local LLM (Qwen2.5-0.5B-Instruct)", "pinned_version": "see requirements.txt", "last_verified": str(date.today()), "recheck_every": "quarterly"},
])
staleness_register

## Exercise

Extend `eval_set` with five more (question, gold-chunk) pairs of your own, re-run recall@k for
k=1..5, and plot the curve. At what k does recall plateau, and what does that tell you about a
reasonable default `k` for this corpus?

## Revision extension: inference metrics and release gate

This extension records synthetic TTFT, total latency, token counts, and an offline release gate for a candidate RAG/LLM deployment.


In [ ]:
requests = [
    {"id": 1, "input_tokens": 420, "output_tokens": 80, "retrieval_ms": 35, "ttft_ms": 480, "total_ms": 1900, "faithful": True},
    {"id": 2, "input_tokens": 900, "output_tokens": 140, "retrieval_ms": 62, "ttft_ms": 760, "total_ms": 3400, "faithful": True},
    {"id": 3, "input_tokens": 610, "output_tokens": 110, "retrieval_ms": 41, "ttft_ms": 530, "total_ms": 2600, "faithful": False},
]
faithfulness = sum(r["faithful"] for r in requests) / len(requests)
avg_ttft = sum(r["ttft_ms"] for r in requests) / len(requests)
avg_total = sum(r["total_ms"] for r in requests) / len(requests)
print({"faithfulness": faithfulness, "avg_ttft_ms": avg_ttft, "avg_total_ms": avg_total})


In [ ]:
gate = {"min_faithfulness": 0.9, "max_avg_ttft_ms": 800, "max_avg_total_ms": 3500}
passes = faithfulness >= gate["min_faithfulness"] and avg_ttft <= gate["max_avg_ttft_ms"] and avg_total <= gate["max_avg_total_ms"]
print("release gate:", "PASS" if passes else "BLOCK")
